<a href="https://colab.research.google.com/github/josephwang02/AAI2025/blob/main/Code%20Generation%20with%20ReACT%20Prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
"""A self-contained, demo travel assistant with richer flight search options.

The assistant uses only the embedded DEMO_FLIGHTS list. It does not call an
external database, booking API, or live airfare service.

It supports maximum budget, route, travel date, passenger count, cabin,
nonstop/max-stops preference, and result sorting.

All fares, schedules, and availability are illustrative teaching data. Verify
current flight information with the airline before booking.
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime


DEMO_DATE = "2026-10-15"


@dataclass(frozen=True)
class Flight:
    airline: str
    flight_number: str
    origin_city: str
    origin_code: str
    destination_city: str
    destination_code: str
    travel_date: str
    departure: str
    arrival: str
    duration_minutes: int
    stops: int
    price: float
    cabin: str
    fare_type: str
    baggage_note: str
    source: str


# Route examples are based on official airline route-map or booking pages.
# Prices and schedules are static demonstration values, not live offers.
DEMO_FLIGHTS = [
    Flight("American Airlines", "AA 1421", "Dallas", "DFW", "Los Angeles", "LAX", DEMO_DATE, "08:10", "09:35", 205, 0, 189.00, "economy", "Demo Economy", "Verify baggage fees", "aa.com"),
    Flight("American Airlines", "AA 1784", "Dallas", "DFW", "Miami", "MIA", DEMO_DATE, "10:25", "14:20", 175, 0, 219.00, "economy", "Demo Economy", "Verify baggage fees", "aa.com"),
    Flight("American Airlines", "AA 2310", "Dallas", "DFW", "New York", "LGA", DEMO_DATE, "13:15", "17:45", 210, 0, 245.00, "economy", "Demo Economy", "Verify baggage fees", "aa.com"),
    Flight("American Airlines", "AA 2678", "Dallas", "DFW", "Chicago", "ORD", DEMO_DATE, "18:05", "20:35", 150, 0, 159.00, "economy", "Demo Economy", "Verify baggage fees", "aa.com"),
    Flight("Delta Air Lines", "DL 418", "Atlanta", "ATL", "Los Angeles", "LAX", DEMO_DATE, "07:30", "09:35", 305, 0, 205.00, "economy", "Demo Economy", "Verify baggage fees", "delta.com"),
    Flight("Delta Air Lines", "DL 1022", "Atlanta", "ATL", "New York", "JFK", DEMO_DATE, "11:20", "13:45", 145, 0, 178.00, "economy", "Demo Economy", "Verify baggage fees", "delta.com"),
    Flight("Delta Air Lines", "DL 1473", "Atlanta", "ATL", "Miami", "MIA", DEMO_DATE, "15:10", "17:05", 115, 0, 142.00, "economy", "Demo Economy", "Verify baggage fees", "delta.com"),
    Flight("United Airlines", "UA 552", "Chicago", "ORD", "Denver", "DEN", DEMO_DATE, "09:00", "10:40", 160, 0, 137.00, "economy", "Demo Economy", "Verify baggage fees", "united.com"),
    Flight("United Airlines", "UA 1847", "Chicago", "ORD", "San Francisco", "SFO", DEMO_DATE, "12:15", "15:00", 285, 0, 198.00, "economy", "Demo Economy", "Verify baggage fees", "united.com"),
    Flight("Southwest Airlines", "WN 731", "Dallas", "DAL", "Denver", "DEN", DEMO_DATE, "06:45", "08:00", 135, 0, 129.00, "economy", "Demo Economy", "Verify baggage rules", "southwest.com"),
    Flight("Southwest Airlines", "WN 1205", "Dallas", "DAL", "Las Vegas", "LAS", DEMO_DATE, "14:25", "15:20", 175, 0, 115.00, "economy", "Demo Economy", "Verify baggage rules", "southwest.com"),
    Flight("Southwest Airlines", "WN 2066", "Dallas", "DAL", "Chicago", "MDW", DEMO_DATE, "17:30", "19:45", 135, 0, 149.00, "economy", "Demo Economy", "Verify baggage rules", "southwest.com"),
]


CITY_ALIASES = {
    "atlanta": {"ATL"},
    "chicago": {"ORD", "MDW"},
    "dallas": {"DFW", "DAL"},
    "dallas love field": {"DAL"},
    "denver": {"DEN"},
    "los angeles": {"LAX"},
    "miami": {"MIA"},
    "new york": {"JFK", "LGA"},
    "san francisco": {"SFO"},
    "las vegas": {"LAS"},
}


def normalize_location(location: str) -> set[str]:
    """Return one or more uppercase airport codes for a city or airport."""
    cleaned = location.strip().lower()
    return CITY_ALIASES.get(cleaned, {cleaned.upper()})


def duration_text(minutes: int) -> str:
    """Format a duration such as 205 minutes as '3h 25m'."""
    hours, remaining_minutes = divmod(minutes, 60)
    return f"{hours}h {remaining_minutes:02d}m"


def search_flights(
    max_price: float,
    beginning: str,
    ending: str,
    travel_date: str | None = None,
    passengers: int = 1,
    cabin: str = "economy",
    nonstop_only: bool = False,
    max_stops: int | None = None,
    sort_by: str = "price",
) -> list[Flight]:
    """Return flights matching the user's preferences."""
    origin_codes = normalize_location(beginning)
    destination_codes = normalize_location(ending)
    normalized_cabin = cabin.strip().lower()

    flights = [
        flight
        for flight in DEMO_FLIGHTS
        if flight.origin_code in origin_codes
        and flight.destination_code in destination_codes
        and flight.price <= max_price
        and (travel_date is None or flight.travel_date == travel_date)
        and flight.cabin == normalized_cabin
        and (not nonstop_only or flight.stops == 0)
        and (max_stops is None or flight.stops <= max_stops)
    ]

    sort_keys = {
        "price": lambda flight: flight.price,
        "duration": lambda flight: flight.duration_minutes,
        "departure": lambda flight: flight.departure,
        "airline": lambda flight: flight.airline,
    }
    return sorted(flights, key=sort_keys.get(sort_by, sort_keys["price"]))


def find_route_alternatives(beginning: str, ending: str) -> list[Flight]:
    """Find route matches without budget and preference filters."""
    origin_codes = normalize_location(beginning)
    destination_codes = normalize_location(ending)
    return sorted(
        (
            flight
            for flight in DEMO_FLIGHTS
            if flight.origin_code in origin_codes
            and flight.destination_code in destination_codes
        ),
        key=lambda flight: flight.price,
    )


def display_flights(
    flights: list[Flight],
    beginning: str,
    ending: str,
    max_price: float,
    travel_date: str | None,
    passengers: int,
    cabin: str,
    sort_by: str,
) -> None:
    """Print a descriptive, travel-assistant-style response."""
    date_label = travel_date or "any date in the demonstration schedule"
    print("\n" + "=" * 72)
    print("TRAVEL ASSISTANT SEARCH SUMMARY")
    print("=" * 72)
    print(f"Route: {beginning.title()} → {ending.title()}")
    print(f"Travel date: {date_label}")
    print(f"Passengers: {passengers} | Cabin: {cabin.title()}")
    print(f"Budget: up to ${max_price:,.2f} per traveler | Sorted by: {sort_by}")
    print("\nDEMO RESULTS — verify live availability and final fare before booking.\n")

    if not flights:
        print("No flights matched all of those preferences.")
        return

    print(f"Found {len(flights)} matching flight(s):\n")
    for index, flight in enumerate(flights, start=1):
        total_price = flight.price * passengers
        stop_text = "Nonstop" if flight.stops == 0 else f"{flight.stops} stop(s)"
        print(f"Option {index}: {flight.airline} {flight.flight_number}")
        print(f"  Route:       {flight.origin_code} → {flight.destination_code}")
        print(f"  Date:        {flight.travel_date}")
        print(f"  Schedule:    {flight.departure}–{flight.arrival} ({duration_text(flight.duration_minutes)})")
        print(f"  Stops:       {stop_text}")
        print(f"  Fare:        ${flight.price:,.2f} per traveler")
        print(f"  Estimated:   ${total_price:,.2f} for {passengers} traveler(s)")
        print(f"  Fare type:   {flight.fare_type}; {flight.baggage_note}")
        print(f"  Verify at:   {flight.source}")
        print()


def read_price() -> float:
    """Prompt until the user enters a valid positive maximum price."""
    while True:
        try:
            price = float(input("Maximum price per traveler (USD): ").strip())
            if price < 0:
                raise ValueError
            return price
        except ValueError:
            print("Please enter a non-negative number, such as 250 or 499.99.")


def read_passengers() -> int:
    """Prompt until the user enters a valid passenger count."""
    while True:
        try:
            passengers = int(input("Number of passengers [1]: ").strip() or "1")
            if passengers < 1:
                raise ValueError
            return passengers
        except ValueError:
            print("Please enter a whole number of at least 1.")


def read_date() -> str | None:
    """Read an optional ISO date, or return None for any demo date."""
    while True:
        value = input(f"Travel date YYYY-MM-DD [any; demo date is {DEMO_DATE}]: ").strip()
        if not value:
            return None
        try:
            datetime.strptime(value, "%Y-%m-%d")
            return value
        except ValueError:
            print("Please use the format YYYY-MM-DD, such as 2026-10-15.")


def read_yes_no(prompt: str, default: bool = False) -> bool:
    """Read a yes/no response with a default."""
    value = input(prompt).strip().lower()
    if not value:
        return default
    return value in {"y", "yes"}


def read_max_stops() -> int | None:
    """Read an optional maximum number of stops."""
    value = input("Maximum stops [any]: ").strip().lower()
    if not value or value in {"any", "none"}:
        return None
    try:
        stops = int(value)
        if stops < 0:
            raise ValueError
        return stops
    except ValueError:
        print("Invalid stop preference; using any number of stops.")
        return None


def read_sort_order() -> str:
    """Read a supported result sort order."""
    value = input("Sort by price, duration, departure, or airline [price]: ").strip().lower()
    return value if value in {"price", "duration", "departure", "airline"} else "price"


def run_travel_assistant() -> None:
    """Run the interactive travel search."""
    print("Travel Assistant")
    print("Search the embedded demonstration flight dataset.\n")

    max_price = read_price()
    beginning = input("Beginning city or airport code: ").strip()
    ending = input("Ending city or airport code: ").strip()
    if not beginning or not ending:
        print("Both a beginning location and an ending location are required.")
        return

    travel_date = read_date()
    passengers = read_passengers()
    cabin = input("Cabin (economy/business) [economy]: ").strip().lower() or "economy"
    nonstop_only = read_yes_no("Nonstop only? (y/n) [n]: ")
    max_stops = read_max_stops()
    sort_by = read_sort_order()

    flights = search_flights(
        max_price=max_price,
        beginning=beginning,
        ending=ending,
        travel_date=travel_date,
        passengers=passengers,
        cabin=cabin,
        nonstop_only=nonstop_only,
        max_stops=max_stops,
        sort_by=sort_by,
    )
    display_flights(
        flights,
        beginning,
        ending,
        max_price,
        travel_date,
        passengers,
        cabin,
        sort_by,
    )

    if not flights:
        alternatives = find_route_alternatives(beginning, ending)
        if alternatives:
            cheapest = alternatives[0]
            print(
                f"\nRoute insight: the lowest embedded fare for this route is "
                f"${cheapest.price:,.2f} per traveler on {cheapest.airline}."
            )
            print("Try increasing the budget or relaxing one of the filters.")
        else:
            print("Route insight: this embedded demo dataset has no record for that route.")


if __name__ == "__main__":
    run_travel_assistant()


Travel Assistant
Search the embedded demonstration flight dataset.

Maximum price per traveler (USD): 500
Beginning city or airport code: SF
Ending city or airport code: london
Travel date YYYY-MM-DD [any; demo date is 2026-10-15]: any
Please use the format YYYY-MM-DD, such as 2026-10-15.
Travel date YYYY-MM-DD [any; demo date is 2026-10-15]: 2026-10-15
Number of passengers [1]: 1
Cabin (economy/business) [economy]: business
Nonstop only? (y/n) [n]: y
Maximum stops [any]: 0
Sort by price, duration, departure, or airline [price]: price

TRAVEL ASSISTANT SEARCH SUMMARY
Route: Sf → London
Travel date: 2026-10-15
Passengers: 1 | Cabin: Business
Budget: up to $500.00 per traveler | Sorted by: price

DEMO RESULTS — verify live availability and final fare before booking.

No flights matched all of those preferences.
Route insight: this embedded demo dataset has no record for that route.
